# Colab RaMF 训练、评估与打包

这个 notebook 专门用于 RaMF 模型性能测试，整体流程尽量和 `colab_unified.ipynb` 保持一致：解压库代码、刷新模块、准备数据、训练、验证集检查、曲线展示和结果打包。

RaMF 会复用原来 `raman` 包里的数据读取、预处理、切分和指标工具，但模型与训练入口放在独立的 `ramf` 包里。

In [ ]:
from pathlib import Path
import shutil
import sys

PROJECT_ROOT = Path.cwd()
RAMAN_ZIP_PATH = PROJECT_ROOT / 'raman.zip'
RAMF_ZIP_PATH = PROJECT_ROOT / 'ramf.zip'
PROJECT_ZIP_PATH = PROJECT_ROOT / 'ramf_project.zip'

# 如果上传的是完整项目压缩包，里面应包含 raman/ 和 ramf/。
RUN_UNPACK_PROJECT = PROJECT_ZIP_PATH.exists()
# 解压前是否清掉旧代码目录。
CLEAR_CODE_BEFORE_UNPACK = True

def remove_code_dir(name):
    target = PROJECT_ROOT / name
    if target.exists():
        shutil.rmtree(target)
        print(f'已删除旧目录：{target}')

if RUN_UNPACK_PROJECT:
    if CLEAR_CODE_BEFORE_UNPACK:
        remove_code_dir('raman')
        remove_code_dir('ramf')
    shutil.unpack_archive(PROJECT_ZIP_PATH, PROJECT_ROOT)
    print(f'已解压完整项目：{PROJECT_ZIP_PATH}')
else:
    if RAMAN_ZIP_PATH.exists():
        if CLEAR_CODE_BEFORE_UNPACK:
            remove_code_dir('raman')
        shutil.unpack_archive(RAMAN_ZIP_PATH, PROJECT_ROOT)
        print(f'已解压：{RAMAN_ZIP_PATH}')
    else:
        print(f'未找到：{RAMAN_ZIP_PATH}')

    if RAMF_ZIP_PATH.exists():
        if CLEAR_CODE_BEFORE_UNPACK:
            remove_code_dir('ramf')
        shutil.unpack_archive(RAMF_ZIP_PATH, PROJECT_ROOT)
        print(f'已解压：{RAMF_ZIP_PATH}')
    else:
        print(f'未找到：{RAMF_ZIP_PATH}')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print('当前目录：', PROJECT_ROOT)


In [ ]:
import sys

# 修改库代码或重新解压后，运行这个单元刷新模块。
def reload_all():
    for name in list(sys.modules):
        if name == 'raman' or name.startswith('raman.'):
            del sys.modules[name]
        if name == 'ramf' or name.startswith('ramf.'):
            del sys.modules[name]
    import raman.config as config_module
    import ramf
    print('模块已刷新。')
    return config_module.config

config = reload_all()


In [ ]:
from dataclasses import asdict, is_dataclass

def config_group_to_dict(group):
    if group is None:
        return {}
    if is_dataclass(group):
        return asdict(group)
    if hasattr(group, 'to_dict'):
        return group.to_dict()
    return {
        key: value
        for key, value in vars(group).items()
        if not key.startswith('_')
    }

def print_config_section(title, data):
    print(f'\n===== {title} =====')
    if not data:
        print('空')
        return
    for key, value in data.items():
        print(f'{key}: {value}')

print_config_section(
    '输入派生值',
    {
        'dataset_root': getattr(config, 'dataset_root', None),
        'in_channels': getattr(config, 'in_channels', None),
        'delta': getattr(config, 'delta', None),
    },
)
print_config_section('共享输入配置', config_group_to_dict(getattr(config, 'shared', None)))
print_config_section('训练默认配置', config_group_to_dict(getattr(config, 'model', None)))


## 数据集配置

这里使用原来 `raman` 的数据集档案。命令行和代码里建议只用英文或数字数据集编号，例如 `GN`、`GP`、`MICRO`。

In [ ]:
from pathlib import Path
import shutil

from raman.data.profiles import get_dataset_dir, get_profile

# 在这里切换数据集。
DATASET_NAME = 'GN'

profile = get_profile(DATASET_NAME)
dataset_dir = get_dataset_dir(profile, PROJECT_ROOT)
dataset_dir.mkdir(parents=True, exist_ok=True)

config.dataset_name = DATASET_NAME
pca_log_path = dataset_dir / profile.pca_log_name
cosmic_log_path = dataset_dir / profile.cosmic_ray_log_name

print('数据集编号 =', config.dataset_name)
print('数据集目录 =', dataset_dir)
print('训练根目录 =', config.dataset_root)
print('坏段配置 =', config.bad_bands)
print('PCA 日志 =', pca_log_path)
print('宇宙射线日志 =', cosmic_log_path)


In [ ]:
from raman.data.io import unpack_init

# 先把 init.npz 上传到当前数据集目录下，再运行这个单元。
pack_path = dataset_dir / profile.root_init_pack
init_dir = dataset_dir / profile.root_init

RUN_UNPACK_INIT = pack_path.exists()
CLEAR_INIT_BEFORE_UNPACK = True

if RUN_UNPACK_INIT:
    if CLEAR_INIT_BEFORE_UNPACK and init_dir.exists():
        shutil.rmtree(init_dir)
        init_dir.mkdir(parents=True, exist_ok=True)
    unpack_init(pack_path, init_dir)
    print('已解压初始数据。')
else:
    print('未找到 init 打包文件，跳过解压。')

print('打包文件 =', pack_path)
print('初始目录 =', init_dir)


In [ ]:
from raman.data.build import build_train

# 从初始目录生成训练集；独立测试集请直接放到对应数据集的测试目录。
RUN_BUILD_TRAIN = True
# 只有想清空旧训练集结果重跑时才打开。
REBUILD_TRAIN = True

train_dir = dataset_dir / profile.root_train_clean
test_dir = dataset_dir / profile.root_test

def reset_dir(path):
    if path.exists():
        shutil.rmtree(path)
    path.mkdir(parents=True, exist_ok=True)

if RUN_BUILD_TRAIN:
    if REBUILD_TRAIN:
        reset_dir(train_dir)
    build_train(profile, dataset_dir)
    print('训练数据已生成。')
else:
    print('跳过训练数据生成。')

print('训练目录 =', train_dir)
print('测试目录 =', test_dir)
print('PCA 日志 =', pca_log_path)
print('宇宙射线日志 =', cosmic_log_path)


In [ ]:
from raman.data.count import count_dataset, print_results

# 默认统计训练集，也可以改成测试集。
COUNT_SUBDIR = 'train'
target_dir = dataset_dir / COUNT_SUBDIR

tree, total_files = count_dataset(target_dir)
print_results(tree, total_files)


## RaMF 训练配置

默认参数按论文结构做了可跑版本：1D Transformer + CFFN、GASF/MTF/RP 三图堆叠、3D-CNN、多头交叉注意力融合。论文没有公开所有层宽细节，所以这里保留可调参数。

In [ ]:
from ramf.config import RaMFConfig
from ramf.train import RaMFTrainConfig

# 目标层级。
TRAIN_LEVEL = 'level_1'

# 训练控制。
EPOCHS = 65
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
PATIENCE = 20
SEED = 42
NUM_WORKERS = 2
USE_GPU = True
# 是否显示批次级进度条。
SHOW_PROGRESS = True

# 输出目录。
OUTPUT_ROOT = '/content/output/ramf'

# 模型参数。
MODEL_DIM = 256
MODEL_HEADS = 8
MODEL_LAYERS = 2
IMAGE_SIZE = 64
MTF_QUANTILE = False
BRANCH_CHANNELS = 32
DROPOUT = 0.5

train_config = RaMFTrainConfig(
    dataset_name=DATASET_NAME,
    level=TRAIN_LEVEL,
    output_root=OUTPUT_ROOT,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    patience=PATIENCE,
    seed=SEED,
    num_workers=NUM_WORKERS,
    use_gpu=USE_GPU,
    show_progress=SHOW_PROGRESS,
)

model_config = RaMFConfig(
    transformer_dim=MODEL_DIM,
    transformer_heads=MODEL_HEADS,
    transformer_layers=MODEL_LAYERS,
    image_size=IMAGE_SIZE,
    mtf_quantile=MTF_QUANTILE,
    branch_channels=BRANCH_CHANNELS,
    dropout=DROPOUT,
)

print('训练配置 =', train_config)
print('模型配置 =', model_config)

In [ ]:
from ramf.train import train_ramf

# 训练完成后会返回本次运行目录，后面的评估和打包都直接复用这个目录。
train_result = train_ramf(train_config, model_config)

RUN_DIR = train_result['run_dir']
BEST_MODEL_PATH = train_result['best_model_path']
RAMF_OUTPUT_ROOT = train_config.output_root

print('RUN_DIR =', RUN_DIR)
print('BEST_MODEL_PATH =', BEST_MODEL_PATH)
train_result

## 结果通用配置

In [ ]:
# 结果入口通用配置
# 如果跳过训练、直接检查已有运行结果，就在这里手动填运行目录。
MANUAL_RUN_DIR = None

if MANUAL_RUN_DIR is not None:
    RUN_DIR = MANUAL_RUN_DIR
elif 'RUN_DIR' not in globals():
    RUN_DIR = None

# 混淆矩阵展示缩放，None 表示按生成图片原始尺寸展示。
CONFUSION_MATRIX_WIDTH = None
CONFUSION_MATRIX_HEIGHT = None

import json
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image, display
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Subset
from tqdm.auto import tqdm

from raman.config import config as eval_config
from raman.data import RamanDataset
from raman.training.split import load_split_files
from ramf.config import RaMFConfig
from ramf.model import RaMFNet


def resolve_current_run_dir():
    if RUN_DIR is None:
        raise ValueError('请填写 MANUAL_RUN_DIR，或先运行训练得到 RUN_DIR')
    run_dir = Path(RUN_DIR)
    if not run_dir.exists() or not run_dir.is_dir():
        raise FileNotFoundError(f'找不到运行目录：{run_dir}')
    return run_dir


def load_json(path):
    with Path(path).open('r', encoding='utf-8') as file:
        return json.load(file)


def load_ramf_model_and_data(run_dir):
    run_dir = Path(run_dir)
    saved_train_config = load_json(run_dir / 'train_config.json')
    saved_model_config = load_json(run_dir / 'ramf_config.json')
    eval_config.dataset_name = saved_train_config['dataset_name']
    dataset = RamanDataset(eval_config.dataset_root, augment=False, config=eval_config)
    level_name = saved_train_config['level']
    level_idx = dataset.head_name_to_idx[level_name]
    classes = dataset.get_class_names(level_name)
    model_config_for_eval = RaMFConfig(**saved_model_config)
    model_config_for_eval.in_channels = int(eval_config.in_channels)
    device = torch.device(
        'cuda'
        if torch.cuda.is_available() and saved_train_config.get('use_gpu', True)
        else 'cpu'
    )
    model = RaMFNet(num_classes=len(classes), config=model_config_for_eval).to(device)
    state = torch.load(run_dir / 'best_model.pt', map_location=device)
    model.load_state_dict(state)
    model.eval()
    return saved_train_config, dataset, level_name, level_idx, classes, model, device


def run_ramf_eval(run_dir=None, result_dir=None, show_progress=True):
    run_dir = Path(run_dir or resolve_current_run_dir())
    result_dir = Path(result_dir or (run_dir / 'val_result'))
    result_dir.mkdir(parents=True, exist_ok=True)
    saved_train_config, dataset, level_name, level_idx, classes, model, device = load_ramf_model_and_data(run_dir)
    split = load_split_files(dataset, run_dir)
    if split is None:
        raise FileNotFoundError(f'运行目录缺少 train_split.json / val_split.json：{run_dir}')
    _, val_idx = split
    loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=int(saved_train_config.get('batch_size', 32)),
        shuffle=False,
        num_workers=0,
    )
    all_paths = []
    all_labels = []
    all_preds = []
    cursor = 0
    iterator = tqdm(loader, desc='验证集预测') if show_progress else loader
    with torch.no_grad():
        for x, y, _ in iterator:
            batch_indices = val_idx[cursor: cursor + x.size(0)]
            cursor += x.size(0)
            x = x.to(device)
            y = y.to(device)
            y_level = y[:, level_idx] if y.ndim == 2 else y
            valid = y_level >= 0
            if not valid.any():
                continue
            logits = model(x)
            preds = logits.argmax(1)
            valid_indices = np.asarray(batch_indices)[valid.detach().cpu().numpy()]
            all_paths.extend([dataset.samples[int(idx)] for idx in valid_indices])
            all_labels.extend(y_level[valid].detach().cpu().numpy().tolist())
            all_preds.extend(preds[valid].detach().cpu().numpy().tolist())

    labels = list(range(len(classes)))
    report_text = classification_report(
        all_labels,
        all_preds,
        labels=labels,
        target_names=classes,
        zero_division=0,
    )
    print(report_text)
    (result_dir / 'classification_report.txt').write_text(report_text, encoding='utf-8')
    pd.DataFrame(
        {
            'path': all_paths,
            'label_true': all_labels,
            'label_pred': all_preds,
        }
    ).to_csv(result_dir / 'val_eval_results.csv', index=False)

    cm = confusion_matrix(all_labels, all_preds, labels=labels)
    pd.DataFrame(cm, index=classes, columns=classes).to_csv(result_dir / 'confusion_matrix_raw.csv')
    fig_size = max(6, min(18, len(classes) * 0.55))
    fig, ax = plt.subplots(figsize=(fig_size, fig_size))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title('RaMF 验证集混淆矩阵')
    ax.set_xlabel('预测类别')
    ax.set_ylabel('真实类别')
    ax.set_xticks(np.arange(len(classes)))
    ax.set_yticks(np.arange(len(classes)))
    ax.set_xticklabels(classes, rotation=90)
    ax.set_yticklabels(classes)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.tight_layout()
    fig.savefig(result_dir / 'confusion_matrix.png', dpi=180)
    plt.close(fig)
    print('val_result_dir =', result_dir)
    return str(result_dir)


def show_confusion_matrix(result_dir, filename='confusion_matrix.png', width=None, height=None):
    result_dir = Path(result_dir)
    cm_path = result_dir / filename
    if not cm_path.exists():
        raise FileNotFoundError(f'找不到混淆矩阵：{cm_path}')
    width = CONFUSION_MATRIX_WIDTH if width is None else width
    height = CONFUSION_MATRIX_HEIGHT if height is None else height
    image_kwargs = {}
    if width is not None:
        image_kwargs['width'] = width
    if height is not None:
        image_kwargs['height'] = height
    display(Image(filename=str(cm_path), **image_kwargs))
    print('confusion_matrix =', cm_path)


def show_training_curves(run_dir=None):
    run_dir = Path(run_dir or resolve_current_run_dir())
    metrics = load_json(run_dir / 'metrics.json')
    history = metrics.get('history', [])
    epochs = [row['epoch'] for row in history]
    train_loss = [row['train']['loss'] for row in history]
    val_loss = [row['val']['loss'] for row in history]
    train_acc = [row['train']['accuracy'] for row in history]
    val_acc = [row['val']['accuracy'] for row in history]
    val_f1 = [row['val']['macro_f1'] for row in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, train_loss, label='训练损失')
    axes[0].plot(epochs, val_loss, label='验证损失')
    axes[0].set_title('损失曲线')
    axes[0].set_xlabel('轮次')
    axes[0].set_ylabel('损失')
    axes[0].legend()
    axes[1].plot(epochs, train_acc, label='训练准确率')
    axes[1].plot(epochs, val_acc, label='验证准确率')
    axes[1].plot(epochs, val_f1, label='验证宏 F1')
    axes[1].set_title('指标曲线')
    axes[1].set_xlabel('轮次')
    axes[1].set_ylabel('分数')
    axes[1].legend()
    plt.tight_layout()
    plt.show()
    print('best_epoch =', metrics.get('best_epoch'))
    print('best_score =', metrics.get('best_score'))


def package_directory(source_dir, output_dir='/content', package_name=None):
    source_dir = Path(source_dir)
    if not source_dir.exists() or not source_dir.is_dir():
        raise FileNotFoundError(f'找不到要压缩的目录：{source_dir}')
    package_name = package_name or source_dir.name
    out_base = Path(output_dir) / package_name
    zip_path = shutil.make_archive(
        base_name=str(out_base),
        format='zip',
        root_dir=str(source_dir.parent),
        base_dir=source_dir.name,
    )
    print('package_source =', source_dir)
    print('package_zip =', zip_path)
    return zip_path


if RUN_DIR is not None:
    print('RUN_DIR =', RUN_DIR)

## 单模型检查

In [ ]:
# 单模型配置
# 填具体运行目录；None 时使用刚训练出来的 RUN_DIR。
SINGLE_RUN_DIR = None

if SINGLE_RUN_DIR is None:
    SINGLE_RUN_DIR = resolve_current_run_dir()
else:
    SINGLE_RUN_DIR = Path(SINGLE_RUN_DIR)

LAST_RESULT_DIR = SINGLE_RUN_DIR
print('SINGLE_RUN_DIR =', SINGLE_RUN_DIR)

In [ ]:
# 单模型评估结果和混淆矩阵展示
single_val_result_dir = run_ramf_eval(
    SINGLE_RUN_DIR,
    show_progress=True,
)
LAST_RESULT_DIR = single_val_result_dir
print('single_val_result_dir =', single_val_result_dir)
show_confusion_matrix(single_val_result_dir)

In [ ]:
# 单模型训练曲线展示
show_training_curves(SINGLE_RUN_DIR)

In [ ]:
# 压缩单次运行文件夹
SINGLE_PACKAGE_OUTPUT_DIR = '/content'
SINGLE_PACKAGE_NAME = None

single_run_zip = package_directory(
    SINGLE_RUN_DIR,
    output_dir=SINGLE_PACKAGE_OUTPUT_DIR,
    package_name=SINGLE_PACKAGE_NAME,
)

## 最后打包

In [ ]:
# 最后压缩整个 RaMF 输出目录
# 日常局部下载可使用前面的单次运行打包单元；这里用于完整保存本数据集的 RaMF 结果。
FINAL_PACKAGE_DIR = Path(RAMF_OUTPUT_ROOT) / DATASET_NAME
FINAL_PACKAGE_OUTPUT_DIR = '/content'
FINAL_PACKAGE_NAME = None

final_zip = package_directory(
    FINAL_PACKAGE_DIR,
    output_dir=FINAL_PACKAGE_OUTPUT_DIR,
    package_name=FINAL_PACKAGE_NAME,
)

In [ ]:
# 最后压缩当前数据集的训练目录
# 保存本次实际使用的预处理训练数据，便于后续复现实验。
TRAIN_PACKAGE_SOURCE_DIR = train_dir
TRAIN_PACKAGE_OUTPUT_DIR = '/content'
TRAIN_PACKAGE_NAME = f'{DATASET_NAME}_train'

train_zip = package_directory(
    TRAIN_PACKAGE_SOURCE_DIR,
    output_dir=TRAIN_PACKAGE_OUTPUT_DIR,
    package_name=TRAIN_PACKAGE_NAME,
)